# Librerias

In [15]:
import pandas as pd
import numpy as np
from kmodes.kmodes import KModes
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns

# Carga de la bbdd y reducción

## 1. Cargar csv

In [2]:
#cargar datos
df = pd.read_parquet("../../data/df_limpiado_260427.parquet")

df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,contactado_campania_previa
0,59,administration,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,NaN,0,sin_campania_previa,yes,0
1,56,administration,married,secondary,no,45,no,no,unknown,5,may,1467,1,NaN,0,sin_campania_previa,yes,0
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,NaN,0,sin_campania_previa,yes,0
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,NaN,0,sin_campania_previa,yes,0
4,54,administration,married,tertiary,no,184,no,no,unknown,5,may,673,2,NaN,0,sin_campania_previa,yes,0


In [3]:
#Nos quedamos con las variables que usaremos para perfil de cliente
cols = ["age", "job", "marital", "education", "balance", "default", "housing", "loan", "deposit"]

df_perfil = df[cols].copy()
df_perfil.info()

<class 'pandas.DataFrame'>
RangeIndex: 10987 entries, 0 to 10986
Data columns (total 9 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   age        10987 non-null  int64 
 1   job        10987 non-null  string
 2   marital    10987 non-null  string
 3   education  10987 non-null  string
 4   balance    10987 non-null  int64 
 5   default    10987 non-null  string
 6   housing    10987 non-null  string
 7   loan       10987 non-null  string
 8   deposit    10987 non-null  string
dtypes: int64(2), string(7)
memory usage: 1.1 MB


## 2. Categorizar edad

In [4]:
df_perfil["age_group"] = pd.cut(df_perfil["age"], bins=[0, 25, 35, 45, 55, 65, 120], labels=["<=25", "26-35", "36-45", "46-55", "56-65", "65+"])

#Eliminamos la columna numérica original
df_perfil = df_perfil.drop(columns=["age"])

## 3. Categorizar multiproducto

In [5]:
#Crear variables binarias 0/1 para cada producto
df_perfil = df_perfil.copy()

df_perfil["deposit_bin"] = (df_perfil["deposit"] == "yes").astype(int)
df_perfil["loan_bin"] = (df_perfil["loan"] == "yes").astype(int)
df_perfil["housing_bin"] = (df_perfil["housing"] == "yes").astype(int)

df_perfil.head(3)

,job,marital,education,balance,default,housing,loan,deposit,age_group,deposit_bin,loan_bin,housing_bin
0,administration,married,secondary,2343,no,yes,no,yes,56-65,1,0,1
1,administration,married,secondary,45,no,no,no,yes,56-65,1,0,0
2,technician,married,secondary,1270,no,yes,no,yes,36-45,1,0,1


In [6]:
#Crear variable combinación, según orden de 0 y 1 se puede saber que productos
def multiproducto(row):
    return f"{row['deposit_bin']}{row['loan_bin']}{row['housing_bin']}"

df_perfil["multiproducto"] = df_perfil.apply(multiproducto, axis=1)

df_perfil.head(3)

,job,marital,education,balance,default,housing,loan,deposit,age_group,deposit_bin,loan_bin,housing_bin,multiproducto
0,administration,married,secondary,2343,no,yes,no,yes,56-65,1,0,1,101
1,administration,married,secondary,45,no,no,no,yes,56-65,1,0,0,100
2,technician,married,secondary,1270,no,yes,no,yes,36-45,1,0,1,101


In [7]:
#Crear columna catagorica de multiproducto:
etiquetas = {
    "000": "Ningún producto",
    "100": "Solo depósitos",
    "010": "Solo préstamos",
    "001": "Solo hipotecas",
    "110": "Depósitos y préstamos",
    "101": "Depósitos y hipotecas",
    "011": "Préstamos y hipotecas",
    "111": "Todos los productos"
}

df_perfil['multiproducto_cat'] = df_perfil['multiproducto'].astype(str).map(etiquetas)

df_perfil.head(3)

,job,marital,education,balance,default,housing,loan,deposit,age_group,deposit_bin,loan_bin,housing_bin,multiproducto,multiproducto_cat
0,administration,married,secondary,2343,no,yes,no,yes,56-65,1,0,1,101,Depósitos y hipotecas
1,administration,married,secondary,45,no,no,no,yes,56-65,1,0,0,100,Solo depósitos
2,technician,married,secondary,1270,no,yes,no,yes,36-45,1,0,1,101,Depósitos y hipotecas


In [8]:
#Crear columna conteo de productos:
df_perfil["cantidad_prod"] = df_perfil["deposit_bin"] + df_perfil["loan_bin"] + df_perfil["housing_bin"]

df_perfil.head(3)

,job,marital,education,balance,default,housing,loan,deposit,age_group,deposit_bin,loan_bin,housing_bin,multiproducto,multiproducto_cat,cantidad_prod
0,administration,married,secondary,2343,no,yes,no,yes,56-65,1,0,1,101,Depósitos y hipotecas,2
1,administration,married,secondary,45,no,no,no,yes,56-65,1,0,0,100,Solo depósitos,1
2,technician,married,secondary,1270,no,yes,no,yes,36-45,1,0,1,101,Depósitos y hipotecas,2


In [9]:
#Crear columna multiproducto binaria:
df_perfil["multiproducto_bin"] = (df_perfil["cantidad_prod"] > 1).astype(int)

df_perfil.head(3)

,job,marital,education,balance,default,housing,loan,deposit,age_group,deposit_bin,loan_bin,housing_bin,multiproducto,multiproducto_cat,cantidad_prod,multiproducto_bin
0,administration,married,secondary,2343,no,yes,no,yes,56-65,1,0,1,101,Depósitos y hipotecas,2,1
1,administration,married,secondary,45,no,no,no,yes,56-65,1,0,0,100,Solo depósitos,1,0
2,technician,married,secondary,1270,no,yes,no,yes,36-45,1,0,1,101,Depósitos y hipotecas,2,1


In [10]:
## 4. Clústers

In [11]:
#dataset para modelo:
cols_modelo = ["age_group", "job", "marital", "education"]
df_perfil_m = df_perfil[cols_modelo].copy()
data_model = df_perfil_m.copy()
X = data_model.to_numpy()

#Entrenar modelo:
k_opt = 5
km = KModes(n_clusters=k_opt, init='Cao', n_init=5, verbose=0)
clusters = km.fit_predict(X)

#Añadir columna clúster a dataset original:
df_perfil["cluster"] = clusters
df_perfil["cluster"].value_counts().sort_index()

cluster
0    5546
1    1758
2    1212
3    1005
4    1466
Name: count, dtype: int64

In [12]:
#Centroides:
centroides = km.cluster_centroids_
centroides

array([['26-35', 'management', 'married', 'secondary'],
       ['36-45', 'technician', 'single', 'tertiary'],
       ['46-55', 'technician', 'married', 'tertiary'],
       ['46-55', 'administration', 'single', 'secondary'],
       ['36-45', 'blue-collar', 'married', 'primary']], dtype='<U14')

In [13]:
# Moda por clúster
multiproducto_moda = (
    df_perfil.groupby("cluster")["multiproducto_cat"]
    .agg(lambda x: x.mode().iloc[0])
    .sort_index()
)

# Centroides del modelo
perfil_clusters = pd.DataFrame(
    km.cluster_centroids_,
    columns=cols_modelo,
    index=[f"Cluster {i}" for i in range(km.n_clusters)]
)

# Añadir moda
perfil_clusters["multiproducto_moda"] = multiproducto_moda.values

perfil_clusters

,age_group,job,marital,education,multiproducto_moda
Cluster 0,26-35,management,married,secondary,Solo depósitos
Cluster 1,36-45,technician,single,tertiary,Solo depósitos
Cluster 2,46-55,technician,married,tertiary,Solo depósitos
Cluster 3,46-55,administration,single,secondary,Solo depósitos
Cluster 4,36-45,blue-collar,married,primary,Solo hipotecas


In [31]:
#Nos quedamos con variables para estudiar productos
cols_prod = ["cluster", "age_group", "job", "marital", "education", "balance","deposit", "loan", "housing", "default", "multiproducto", "multiproducto_cat", "cantidad_prod", "multiproducto_bin"]

df_product = df_perfil[cols_prod].copy()

df_product.head(3)

,cluster,age_group,job,marital,education,balance,deposit,loan,housing,default,multiproducto,multiproducto_cat,cantidad_prod,multiproducto_bin
0,0,56-65,administration,married,secondary,2343,yes,no,yes,no,101,Depósitos y hipotecas,2,1
1,0,56-65,administration,married,secondary,45,yes,no,no,no,100,Solo depósitos,1,0
2,0,36-45,technician,married,secondary,1270,yes,no,yes,no,101,Depósitos y hipotecas,2,1


## 5. Análisis descriptivo

## 6. Regresión logística binaria

In [33]:
X = df_product[['cluster','age_group','job','marital','education',
                'balance','deposit','loan','housing','default']]

# Convertir categóricas a dummies
X = pd.get_dummies(X, drop_first=True)

# Asegurar que todo sea numérico
X = X.apply(pd.to_numeric, errors='coerce')

# Eliminar filas con NA
X = X.dropna()
y = df_product.loc[X.index, 'multiproducto_bin']

# Agregar constante
X = sm.add_constant(X)

modelo = sm.Logit(y, X).fit()
print(modelo.summary())

ValueError: Pandas data cast to numpy dtype of object. Check input data with np.asarray(data).

## 7. Conclusiones